<img src="https://govspace.io/wp-content/uploads/2025/09/GovSpace-web.svg" width="200"># Module M2.9 — Tool-using agents and multi-step workflowsThis super-notebook is the hands-on companion to PDS Module 2.9. You will work through three connected pieces, in one continuous arc, in about 40 minutes:1. **Setup** — paste your API key and confirm the SDKs are installed (~5 min)2. **Basic tool-using agent** — define tools the model can call, run the agent loop on three example questions (~15 min)3. **Multi-step LangGraph workflow** — compile a directed graph with shared state, conditional routing, and a loop-back path; run it end-to-end (~20 min)The whole sequence is the foundational orchestration pattern that **M2.10** layers a governance overlay onto (approval gates, confidence-based routing, audit logging).> **💡 Tip:** Before you change anything, copy this notebook into a folder with your name on it (e.g. `matt-grasser/`). The `TEMPLATES` folder syncs from GitHub and is read-only.> **🔑 Before you start:** running the code below requires an API key from Anthropic or OpenAI. If you do not have one yet, see the *Getting an API Key* guide in the Module 2.9 Moodle section before going further. If you would rather not configure a paid key right now, you can still read each cell as illustration — the code, comments, and expected behaviour are all visible without running.---## Part 1 — Setup

### Step 1: Configure API KeysThis is a **shared server** — all participants can see each other's files and notebook outputs. To keep your API keys safe:1. Paste your keys in the cell below and run it (this sets them in memory only)2. **Immediately clear the cell output** after running: right-click the cell output → "Clear Outputs"3. The keys stay in memory for this session but are not visible to others

In [ ]:
import os

# ✏️ Replace the placeholder values with your real API keys, then run this cell.
# ⚠️ CLEAR THE OUTPUT immediately after running (right-click output → Clear Outputs).

os.environ["ANTHROPIC_API_KEY"] = "sk-ant-PASTE-YOUR-KEY-HERE"
os.environ["OPENAI_API_KEY"] = "sk-PASTE-YOUR-KEY-HERE"

# Verify keys are loaded (shows first/last few chars only)
for key in ["ANTHROPIC_API_KEY", "OPENAI_API_KEY"]:
    val = os.environ.get(key, "")
    if val and "PASTE" not in val:
        print(f"✅ {key} loaded ({val[:8]}...{val[-4:]})")
    else:
        print(f"❌ {key} not set — replace the placeholder above and re-run")

### Step 2: Verify the SDKsThe Agentic Gym ships with the Anthropic, OpenAI, LangChain, and LangGraph SDKs pre-installed. Run this to confirm everything is loaded.

In [ ]:
import importlib

packages = [
    ("anthropic", "Anthropic SDK"),
    ("openai", "OpenAI SDK"),
    ("langchain_core", "LangChain Core"),
    ("langchain_anthropic", "LangChain Anthropic"),
    ("langchain_openai", "LangChain OpenAI"),
    ("langgraph", "LangGraph"),
    ("httpx", "httpx"),
    ("dotenv", "python-dotenv"),
]

for module, name in packages:
    try:
        mod = importlib.import_module(module)
        version = getattr(mod, "__version__", "installed")
        print(f"✅ {name}: {version}")
    except ImportError:
        print(f"❌ {name}: NOT INSTALLED")

### Step 3: Hello-world callsOne call to Claude, one to GPT. If both return a sentence, your environment is ready.

In [ ]:
import anthropic

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env

message = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=256,
    messages=[{"role": "user", "content": "Say hello in exactly one sentence."}],
)

print(message.content[0].text)

In [ ]:
from openai import OpenAI

client = OpenAI()  # reads OPENAI_API_KEY from env

response = client.chat.completions.create(
    model="gpt-4o-mini",
    max_tokens=256,
    messages=[{"role": "user", "content": "Say hello in exactly one sentence."}],
)

print(response.choices[0].message.content)

---## Part 2 — A tool-using agentA plain language-model call is a closed conversation: you ask, it answers, you're done. An **agent** is the same model wrapped in a small loop that lets it call **tools** — Python functions you expose with a JSON schema — see the results, and decide what to do next.In this part you will build a tool-using agent with two tools (a calculator and a weather lookup) and watch the model decide when to call which one. The pattern is short:```send message + tool list → if model wants tool: run handler, append result → repeat → until model returns text```That loop is the foundation. Everything in Part 3 (LangGraph) is a way to structure this same agentic logic across multiple specialised steps.

In [ ]:
import os
import anthropic

# Keys should already be set from 00-Setup. If not, set them here:
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

client = anthropic.Anthropic()

## Step 1: Define Tools

Tools are JSON schemas that tell the model what functions are available and what arguments they accept.

Let's create two simple tools: a calculator and a weather lookup.

In [ ]:
tools = [
    {
        "name": "calculate",
        "description": "Evaluate a mathematical expression. Use this for any arithmetic.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "A mathematical expression to evaluate, e.g. '(25 * 4) + 10'"
                }
            },
            "required": ["expression"]
        }
    },
    {
        "name": "get_weather",
        "description": "Get the current weather for a city.",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "City name, e.g. 'Ottawa'"
                }
            },
            "required": ["city"]
        }
    }
]

## Step 2: Implement Tool Handlers

These are the actual Python functions that run when the model calls a tool.

In [ ]:
def handle_tool_call(tool_name: str, tool_input: dict) -> str:
    """Dispatch a tool call to the appropriate handler."""
    if tool_name == "calculate":
        try:
            # Safety note: in production, use a proper math parser, not eval()
            result = eval(tool_input["expression"])
            return str(result)
        except Exception as e:
            return f"Error: {e}"

    elif tool_name == "get_weather":
        # Simulated weather data (in a real app, call a weather API)
        fake_weather = {
            "ottawa": "☀️ 22°C, sunny",
            "toronto": "🌤️ 19°C, partly cloudy",
            "vancouver": "🌧️ 14°C, rain",
        }
        city = tool_input["city"].lower()
        return fake_weather.get(city, f"🌡️ 20°C, weather data not available for {tool_input['city']}")

    return f"Unknown tool: {tool_name}"

## Step 3: The Agent Loop

The core pattern: send a message → if the model wants to call a tool → execute it → feed the result back → repeat until the model responds with text.

This is the **agentic loop** — the model decides what to do next.

In [ ]:
def run_agent(user_message: str, max_turns: int = 5) -> str:
    """Run a simple tool-use agent loop."""
    messages = [{"role": "user", "content": user_message}]

    for turn in range(max_turns):
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=1024,
            tools=tools,
            messages=messages,
        )

        # If the model just responds with text, we're done
        if response.stop_reason == "end_turn":
            text_blocks = [b.text for b in response.content if b.type == "text"]
            return "\n".join(text_blocks)

        # If the model wants to use tools, execute them
        if response.stop_reason == "tool_use":
            # Add the assistant's response (with tool_use blocks) to messages
            messages.append({"role": "assistant", "content": response.content})

            # Process each tool call
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  🔧 Calling: {block.name}({block.input})")
                    result = handle_tool_call(block.name, block.input)
                    print(f"  📎 Result: {result}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result,
                    })

            # Feed results back to the model
            messages.append({"role": "user", "content": tool_results})

    return "Agent reached max turns without completing."

## Step 4: Try It Out

Ask the agent questions that require tool use:

In [ ]:
# A question that requires the calculator
print("--- Calculator ---")
result = run_agent("What is 47 * 89 + 123?")
print(f"\n{result}")

In [ ]:
# A question that requires weather lookup
print("--- Weather ---")
result = run_agent("What's the weather like in Ottawa and Vancouver?")
print(f"\n{result}")

In [ ]:
# A question that requires BOTH tools
print("--- Multi-tool ---")
result = run_agent(
    "If it's 22°C in Ottawa, what is that in Fahrenheit? "
    "Also check: what's the actual weather in Toronto?"
)
print(f"\n{result}")

---## Part 3 — A multi-step workflow with LangGraphThe agent loop you just built handles one model + one tool list in one continuous conversation. Real supervisory tasks usually have **phases** — research, evaluation, synthesis, drafting — with the ability to loop back when a phase didn't produce enough. **LangGraph** lets you compose those phases as a directed graph: nodes are processing steps, edges define the flow, and shared state flows between them.In this part you will build a worked **Research → Evaluate → Summarize** workflow that loops back to Research when more material is needed. Same agentic logic as Part 2, structured across specialised steps.

In [ ]:
import os
from typing import Annotated, TypedDict
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

# Keys should already be set from 00-Setup. If not, set them here:
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

## Concept: Graphs as Workflows

LangGraph lets you model agent workflows as **directed graphs**:

```
[Research] → [Evaluate] → [Summarize] → END
                 ↓
            [Research]  (loop back if more research needed)
```

Each **node** is a function that transforms state. **Edges** define the flow.

## Step 1: Define State

State is a typed dictionary that flows through every node in the graph.

In [ ]:
class ResearchState(TypedDict):
    """State that flows through our research workflow."""
    topic: str
    messages: Annotated[list, add_messages]
    research_notes: list[str]
    research_rounds: int
    summary: str

## Step 2: Define Nodes

Each node is a function that takes state and returns updated state.

In [ ]:
model = ChatAnthropic(model="claude-sonnet-4-20250514", max_tokens=1024)


def research_node(state: ResearchState) -> dict:
    """Generate research notes on the topic."""
    round_num = state.get("research_rounds", 0) + 1
    existing_notes = state.get("research_notes", [])

    if existing_notes:
        context = "\n".join(f"- {n}" for n in existing_notes)
        prompt = (
            f"You are researching: {state['topic']}\n\n"
            f"Previous findings:\n{context}\n\n"
            f"Find 2-3 NEW angles or details not yet covered. "
            f"Be specific and factual. Return only bullet points."
        )
    else:
        prompt = (
            f"You are researching: {state['topic']}\n\n"
            f"Provide 3-4 key findings as bullet points. "
            f"Be specific and factual."
        )

    response = model.invoke([HumanMessage(content=prompt)])
    new_notes = [line.strip("- ").strip() for line in response.content.split("\n") if line.strip().startswith("-")]

    print(f"📚 Research round {round_num}: found {len(new_notes)} points")
    for note in new_notes:
        print(f"   • {note[:80]}..." if len(note) > 80 else f"   • {note}")

    return {
        "research_notes": existing_notes + new_notes,
        "research_rounds": round_num,
        "messages": [response],
    }


def evaluate_node(state: ResearchState) -> dict:
    """Decide if we have enough research or need another round."""
    notes = state.get("research_notes", [])
    rounds = state.get("research_rounds", 0)

    print(f"\n🔍 Evaluating: {len(notes)} notes after {rounds} round(s)")
    # Simple heuristic: do 2 rounds, then move to summary
    return state


def summarize_node(state: ResearchState) -> dict:
    """Synthesize research notes into a coherent summary."""
    notes = state.get("research_notes", [])
    context = "\n".join(f"- {n}" for n in notes)

    prompt = (
        f"You researched: {state['topic']}\n\n"
        f"Research notes:\n{context}\n\n"
        f"Write a concise 2-3 paragraph summary synthesizing these findings. "
        f"Highlight the most important insights."
    )

    response = model.invoke([HumanMessage(content=prompt)])
    print(f"\n📝 Summary generated ({len(response.content)} chars)")

    return {
        "summary": response.content,
        "messages": [response],
    }

## Step 3: Define Routing Logic

Conditional edges let the graph decide where to go next based on state.

In [ ]:
def should_continue_research(state: ResearchState) -> str:
    """Decide: do more research or move to summary?"""
    rounds = state.get("research_rounds", 0)
    if rounds < 2:
        print("   → More research needed")
        return "research"  # loop back
    else:
        print("   → Enough research, moving to summary")
        return "summarize"  # move forward

## Step 4: Build the Graph

In [ ]:
# Create the graph
workflow = StateGraph(ResearchState)

# Add nodes
workflow.add_node("research", research_node)
workflow.add_node("evaluate", evaluate_node)
workflow.add_node("summarize", summarize_node)

# Define edges
workflow.set_entry_point("research")
workflow.add_edge("research", "evaluate")
workflow.add_conditional_edges(
    "evaluate",
    should_continue_research,
    {"research": "research", "summarize": "summarize"},
)
workflow.add_edge("summarize", END)

# Compile
app = workflow.compile()

print("Graph compiled! Nodes:", list(app.get_graph().nodes.keys()))

## Step 5: Run the Workflow

In [ ]:
result = app.invoke({
    "topic": "How AI agents are being used in government technology (govtech)",
    "messages": [],
    "research_notes": [],
    "research_rounds": 0,
    "summary": "",
})

print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)
print(result["summary"])

## 🧪 Try It Yourself

Change the topic and run again:

In [ ]:
# Try your own topic!
result = app.invoke({
    "topic": "YOUR TOPIC HERE",
    "messages": [],
    "research_notes": [],
    "research_rounds": 0,
    "summary": "",
})

print("\n" + "="*60)
print(result["summary"])

---## 💡 Key takeaways across the whole module1. **Tools are schemas; handlers are Python.** You describe what's available; the model decides when to use it. Swap the implementation behind the schema without changing the contract.2. **The agent loop is small.** Send → tool? execute and feed back → repeat. Twenty lines of code.3. **LangGraph is for when one loop is not enough.** When work has phases, intermediate "good enough?" checks, and accumulating state, model it as a graph instead of a monolithic prompt.4. **State is the audit surface.** Whatever flows through the graph is what you can inspect. Design state up front; treat it as a case-management record.5. **Routing functions are where supervisory policy lives.** The conditional edge is plain Python you can read and edit. M2.10 takes this exact surface and adds human approval gates.---## You have reached the end of this super-notebookReturn to **M2.9** in the GovSpace Academy to:- Post your Discussion Question response in the GovSpace Connect thread- Complete the Module Quiz- Map one workflow in your own agency onto the patterns you just builtThen continue to **M2.10** — Graduated autonomy and human-in-the-loop agent design — which adds approval gates, confidence-based routing, and audit logging on top of the LangGraph foundation you just built.